In [ ]:
# 구버전 신버전은 로레벨 작업이 아닌 이상 session을 통해 객체 생성후 연결
# from pyspark import SparkConf, SparkContext

# spark_conf = SparkConf().setAppName("spark-context-test")
# spark = SparkContext(conf = spark_conf)

In [3]:
# spark context를 포함한 wrapper 객체, high-level 구현
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local')
    .appName('spark-session-test')
    .config('spark.some.config.option','some-value')
    .getOrCreate()
)

In [ ]:
# data file을 읽거나 python data model을 spark data model로 변환
# df = spark.read.json("path/to/json/file")
# df.printSchema()

In [5]:
# 원하는 포맷의 데이터 파일로 저장
# df.wirte \
#     .option('compression','gzip') \
#     .mode('Overwrite') \
#     .json('/path/to/output.json')

In [6]:
print(spark.sparkContext.sparkUser())

jovyan


In [7]:
!whoami

jovyan


In [9]:
PATH = "/home/jovyan/work/SPARK/00/행정안전부_착한가격업소 현황_20260630.csv"

df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")     # 깨지면 UTF-8로 교체
    .option("multiLine", True)
    .option("escape", '"')
    .csv(PATH))

df.printSchema()
print("행 수:", df.count())
print(df.columns)

root
 |-- 시도: string (nullable = true)
 |-- 시군: string (nullable = true)
 |-- 업종: string (nullable = true)
 |-- 업소명: string (nullable = true)
 |-- 연락처: string (nullable = true)
 |-- 주소: string (nullable = true)
 |-- 메뉴1: string (nullable = true)
 |-- 가격1: string (nullable = true)
 |-- 메뉴2: string (nullable = true)
 |-- 가격2: string (nullable = true)
 |-- 메뉴3: string (nullable = true)
 |-- 가격3: string (nullable = true)
 |-- 메뉴4: string (nullable = true)
 |-- 가격4: string (nullable = true)

행 수: 12645
['시도', '시군', '업종', '업소명', '연락처', '주소', '메뉴1', '가격1', '메뉴2', '가격2', '메뉴3', '가격3', '메뉴4', '가격4']


In [10]:
df.show(3, False)

+----------+------+----------+-----------+-----------+--------------------------------------+-----------+-----+-----------+-----+-----+-----+-----+-----+
|시도      |시군  |업종      |업소명     |연락처     |주소                                  |메뉴1      |가격1|메뉴2      |가격2|메뉴3|가격3|메뉴4|가격4|
+----------+------+----------+-----------+-----------+--------------------------------------+-----------+-----+-----------+-----+-----+-----+-----+-----+
|서울특별시|종로구|양식      |돈까스보라 |02-741-3455|서울특별시 종로구 대학로5길 5 (연건동)|수제 돈까스|7000 |돈까스 정식|8000 |NULL |NULL |NULL |NULL |
|서울특별시|종로구|한식      |삼삼 뚝배기|02-765-4683|서울특별시 종로구 동숭길 51 (동숭동)  |된장뚝배기 |7500 |김치찌개   |7500 |NULL |NULL |NULL |NULL |
|서울특별시|종로구|기타요식업|약속커피숍 |02-764-1100|서울특별시 종로구 종로 302 (창신동)   |맥심커피   |2000 |녹차       |2000 |NULL |NULL |NULL |NULL |
+----------+------+----------+-----------+-----------+--------------------------------------+-----------+-----+-----------+-----+-----+-----+-----+-----+
only showing top 3 rows



In [11]:
import pyspark.sql.functions as F

df_u = df.where(F.col('업종')=='양식')
df_u.write.mode('Overwrite').csv('/home/jovyan/work/SPARK/00/output.csv')